In [1]:
import os
HOME = os.path.expanduser("~") + '/gdrive/DataHII/'

In [2]:
import os
import sys
import glob
from pathlib import Path
from astropy.io import fits
import numpy as np
from numpy import savetxt
import matplotlib
import matplotlib.pyplot as plt
from PyAstronomy import pyasl
from scipy.interpolate import interp1d
from astropy import units as u
from astropy.coordinates import SkyCoord
#install dustmaps from https://github.com/gregreen/dustmaps
from dustmaps.sfd import SFDQuery
import pandas as pd
from astropy.table import Table
import gc
matplotlib.use("Agg")
plt.ioff()

In [6]:
tex_folder = '/HIIGs/Tex_spectra/'
plots_folder =  '/HIIGs/Spectra_plots/'

if (os.path.exists(HOME + tex_folder) == True and os.path.exists(HOME + plots_folder) == True):
    print('Folder for TEX set saving already created \n')
    print('Folder for PLOTS set saving already created \n')
else:
    if (os.path.exists(HOME + plots_folder) == False):
        print(f'Folder for PLOTS set saving created ({HOME+plots_folder}) \n')
        os.mkdir(HOME+plots_folder)
    if (os.path.exists(HOME + tex_folder) == False):
        print(f'Folder for TEX set saving created ({HOME+tex_folder}) \n')
        os.mkdir(HOME+plots_folder)
    else:
        print(f'Folder for TEX set saving created ({HOME+tex_folder}) \n')
        print(f'Folder for PLOTS set saving created ({HOME+plots_folder}) \n')
        os.mkdir(HOME+tex_folder)
        os.mkdir(HOME+plots_folder)

Folder for TEX set saving already created 

Folder for PLOTS set saving already created 



In [7]:
fits_folder = HOME + '/HIIGs/FITS/'
spec_list = os.listdir(fits_folder)
spec_list = sorted(spec_list)

In [8]:
def sdss_dust_correction(NAME,counter):
    counter = int(counter)
    hdu = fits.open(fits_folder+NAME)
    data = hdu[0].data
    ##############################################################
    #Read the header
    
    zz = hdu[0].header['Z']
    RA = hdu[0].header['RA']
    DEC = hdu[0].header['DEC']
    
    c = SkyCoord(ra=RA*u.degree, dec=DEC*u.degree, frame='icrs')
    c.galactic
    
    sfd = SFDQuery()
    ebv01 = sfd(c.galactic)
    
    ##############################################################
    #Read the spectra, flux and wavelength
    
    flux = data[0]
    wav00 = hdu[0].header['COEFF0']
    wavdiff = hdu[0].header['COEFF1']

    hdu.close()
    
    waveair = 10**(wav00 + wavdiff*np.arange(len(flux)))
    
    waveobs = waveair/(1.0+2.735182E-4+131.4182/(waveair)**2 +2.76249E8/(waveair)**4)
    
    ##############################################################
    #redshift correction
    
    wave = waveobs/(1.0+zz)
    
    ##############################################################
    #Unredding the spectra
    
    fluxUnred = pyasl.unred(wave, flux, ebv=ebv01, R_V=3.1)
    
    ##############################################################
    #cosmoiogical flux corrextion
    
    flux_final = fluxUnred*(1.0+zz)**3
    
    ##############################################################
    #Change the wavelength to starlight input
    
    wave_in = int(wave[2])             #initial wavelength
    wave_fin = int(wave[len(wave)-2])  #final wavelength
    
    wave_new = np.arange(wave_in, wave_fin, 1)
    
    flux_interpolate = interp1d(wave, flux_final, kind='quadratic')
    flux_starlight = flux_interpolate(wave_new)
    
    ##############################################################
    #Crate an output array
    
    table = list(zip(wave_new, flux_starlight))
    
    # Save the file
    new_name  = HOME + tex_folder  + (NAME.replace('.fit',"")) + '._' + str(counter)+ '_'
    savetxt(new_name + '.tex', table, fmt='%8.4f', delimiter='    ')
    print(NAME+ ' spectra saved in '+new_name + '.tex')
    

In [9]:
for i in range(len(spec_list)):
    sdss_dust_correction(sorted(spec_list)[i],i+1)

spSpec-51608-0267-421.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51608-0267-421._1_.tex
spSpec-51662-0284-277.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51662-0284-277._2_.tex
spSpec-51662-0308-081.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51662-0308-081._3_.tex
spSpec-51690-0341-606.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51690-0341-606._4_.tex
spSpec-51793-0388-457.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51793-0388-457._5_.tex
spSpec-51810-0415-141.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51810-0415-141._6_.tex
spSpec-51811-0381-370.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51811-0381-370._7_.tex
spSpec-51816-0382-328.fit spectra saved in /home/hollman/gdrive/DataHII//HIIGs/Tex_spectra/spSpec-51816-0382-328._8_.tex
spSpec-51816-0410-220.fit spectr

In [10]:
# Opcional: evita trazados gigantes en Agg con paths muy largos
plt.rcParams['agg.path.chunksize'] = 10000

DATA_URL = "http://das.sdss.org/spectro/ss_tar_26/"
catalog_name = "gal_info_dr7_v5_2.fit.gz"
local_dir = HOME

# Lee el catálogo una sola vez, con mapeo de memoria
with fits.open(os.path.join(local_dir, catalog_name), memmap=True) as hdul:
    catalog = Table(hdul[1].data)

ll = ['HB','O[III]a','O[III]b','Ha']
lines = {
  "HB": [4862.721, r'H-$\beta$', 4862.0000],
  "O[III]a": [4960.295, r'O[III]$\lambda$4959', 4960.0000],
  "O[III]b": [5008.239, r'O[III]$\lambda$5007', 5007.0000],
  "Ha": [6562.801, r'H-$\alpha$', 6562.0000]
}

In [11]:
def spec_plotter(name,
                 figsize=(48, 18),        # << mucho más pequeño
                 render_dpi=100,          # DPI de canvas
                 save_dpi=100,            # DPI del archivo final
                 tol=1.0,counter=0):                # tolerancia en Å para localizar líneas

    # Construye paths/nombres como en tu versión
    base = (name.replace('.fit', '')) + ".tex"
    base = base.replace(".tex", f"._{str(counter)}_.tex")
    local_dirspc = HOME + tex_folder
    spc = os.path.join(local_dirspc, base)

    # Extrae IDs desde el nombre, como en tu código
    plateid, mjd, fiberid = [int(base[13:17]), int(base[7:12]), int(base[18:21])]

    # Selección de fila en el catálogo
    z_finder = (catalog['PLATEID'] == plateid) & (catalog['MJD'] == mjd) & (catalog['FIBERID'] == fiberid)
    reduced_cat = catalog[z_finder]
    Z = float(reduced_cat['Z'][0])

    # Lee espectro (rápido y sobrio en memoria)
    data = Table.read(spc, format='ascii', fast_reader=True)
    lam = np.asarray(data['col1'], dtype=float)
    flux = np.asarray(data['col2'], dtype=float)

    # Prepara figura/axes (OO) y estilos básicos
    fig, ax = plt.subplots(figsize=figsize, dpi=render_dpi)
    ax.spines.right.set_visible(False)
    ax.spines.top.set_visible(False)

    # Trazado principal
    ax.plot(lam, flux, label='Flux', lw=1.2)

    ax.set_ylabel(r"$F_{\lambda}\ \left[10^{-17}\ {\rm erg\ s}^{-1}\ {\rm cm}^{-2}\ \AA^{-1}\right]$",
                  style='oblique', family='serif')
    ax.set_xlabel(r'Wavelength at rest $\lambda$ [$\AA$]',
                  style='oblique', family='serif')

    # Anotación de líneas (con tolerancia en lugar de igualdad exacta)
    for key in ll:
        rest = lines[key][2]
        idx = np.where(np.abs(lam - rest) <= tol)[0]
        if idx.size:
            i0 = idx[len(idx)//2]
            i1, i2 = max(i0 - 10, 0), min(i0 + 10, flux.size)
            selec_flux_max = float(flux[i1:i2].max()) + 0.05 * (flux.max() - flux.min())
            x = rest - (100 if key in ('O[III]a', 'O[III]b') else 30)
            ax.text(x, selec_flux_max, f"{lines[key][1]}",
                    fontsize=11,
                    bbox={'facecolor': '#F4F1BB', 'alpha': 0.5, 'boxstyle': "round,pad=0.2", 'ec': 'none'})

    ax.tick_params(axis='x', labelrotation=90)
    ax.set_xlim(3500, 8000)
    ax.set_ylim(0, max(float(flux.max()), 1.0) * 1.05)
    ax.grid(True, which="both", ls=":", linewidth=0.8)

    leg_title = f"{base[:-4]}\nZ = {Z}\nPLATEID = {plateid}\nMJD = {mjd}\nFIBERID = {fiberid}"
    ax.legend(title=leg_title, loc='upper right', prop={'size': 10}, title_fontsize=11)

    outdir = HOME + plots_folder
    os.makedirs(outdir, exist_ok=True)
    outfile = os.path.join(outdir, base.replace('.tex', '') + ".png")

    # Guarda (si "tight" no es imprescindible, quítalo para ahorrar un poco más)
    fig.savefig(outfile, dpi=save_dpi, bbox_inches='tight', transparent=False)

    # Cierra y limpia SOLO esta figura
    plt.close(fig)
    del fig, ax, data, lam, flux, reduced_cat
    gc.collect()
    print('Done for ' + base.replace('.tex', ''))



# Bucle principal (usa directamente la lista)
for h in range(len(spec_list)):
    try:
        spec_plotter(name = sorted(spec_list)[h],counter = h+1)
    except Exception as e:
        print(f"[WARN] Falló {name}: {e}")

Done for spSpec-51608-0267-421._1_
Done for spSpec-51662-0284-277._2_
Done for spSpec-51662-0308-081._3_
Done for spSpec-51690-0341-606._4_
Done for spSpec-51793-0388-457._5_
Done for spSpec-51810-0415-141._6_
Done for spSpec-51811-0381-370._7_
Done for spSpec-51816-0382-328._8_
Done for spSpec-51816-0410-220._9_
Done for spSpec-51820-0400-441._10_
Done for spSpec-51820-0429-495._11_
Done for spSpec-51821-0384-281._12_
Done for spSpec-51877-0447-361._13_
Done for spSpec-51882-0442-156._14_
Done for spSpec-51884-0418-319._15_
Done for spSpec-51908-0481-483._16_
Done for spSpec-51909-0455-073._17_
Done for spSpec-51909-0485-550._18_
Done for spSpec-51910-0275-445._19_
Done for spSpec-51910-0456-195._20_
Done for spSpec-51910-0465-524._21_
Done for spSpec-51914-0488-439._22_
Done for spSpec-51924-0459-253._23_
Done for spSpec-51929-0458-185._24_
Done for spSpec-51929-0490-128._25_
Done for spSpec-51955-0472-546._26_
Done for spSpec-51957-0502-007._27_
Done for spSpec-51959-0283-389._28_
D

In [ ]:
FLAGS = [2,29,72,122,123,124,126,127,128]

# Version vieja no optimizada

In [6]:
DATA_URL="http://das.sdss.org/spectro/ss_tar_26/"
catalog_name="gal_info_dr7_v5_2.fit.gz"
local_dir="/home/hollman/DataHII/"
local_file = fits.open(os.path.join(local_dir,catalog_name))
catalog=Table.read(local_file[1])
local_file.close()

ll = ['HB','O[III]a','O[III]b','Ha']
# Lines dictionary
lines = {
  "HB": [4862.721,r'H-$\beta$',4862.0000], #4862
  "O[III]a": [4960.295,r'O[III]$\lambda$4959',4960.0000],#4960
  "O[III]b": [5008.239,r'O[III]$\lambda$5007',5007.0000],#5007
  "Ha": [6562.801,r'H-$\alpha$',6562.0000] ##6550
}


In [8]:
def spec_plotter(name):
    name = (name.replace('.fit',"")) + ".tex"
    local_dirspc="/home/hollman/DataHII/HIIGalaxy_Spectra(Chavez2012)/Tex_spectra/"
    spc = os.path.join(local_dirspc,name)
    plateid, mjd, fiberid = [int(name[13:17]), int(name[7:12]), int(name[18:21])]
    z_finder = (catalog['PLATEID']==plateid) & (catalog['MJD']==mjd) & (catalog['FIBERID']==fiberid)
    reduced_cat = catalog[z_finder]
    Z = round(float(reduced_cat['Z'][0]),6)
    data = Table.read(spc, format='ascii')
    data.rename_column('col1', r'$\lambda$')
    data.rename_column('col2', 'Flux')
    plt.rcParams['axes.spines.left'] = True
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.bottom'] = True
    plt.figure(figsize=(40,18),dpi=200)
    plt.plot(data[r'$\lambda$'],data['Flux'],label = 'Flux',lw=1.3, color ='#1C448E')
    plt.ylabel(r"$F_{\lambda}\ \left[ 10^{-17}\ {\rm erg\ s}^{-1}\ {\rm cm}^{-2}\ \AA^{-1} \right]$", style = 'oblique', family = 'serif', size = 25)
    plt.xlabel(r'Wavelength at rest $\lambda$ [$\AA$]', style = 'oblique', family = 'serif', size = 25)
    # Lineas a mostrar
    for a in range(len(ll)):#len(ll)
        v_lambda = lines[ll[a]][2] 
        flux_line = data[(data[r'$\lambda$'])==v_lambda]
        selec_flux_idx = np.where(data[r'$\lambda$'] == v_lambda)
        selec_flux_max = max(data['Flux'][(selec_flux_idx[0][0])-10:(selec_flux_idx[0][0])+10]) + 70
        if (a==1) or (a==2):
            v_lambda = v_lambda- 100
        else:
            v_lambda = v_lambda - 30
        plt.text(v_lambda, selec_flux_max, f'{lines[ll[a]][1]}', 
                fontsize = 25,bbox = {'facecolor': '#F4F1BB', 'alpha': 0.5, 'boxstyle': "round,pad=0.3", 'ec': 'none'}, rotation = 0)
    plt.yticks(fontsize = 20)
    plt.xticks(fontsize = 20,rotation=90)
    plt.ylim(0,max(data['Flux'],)+15)
    plt.xlim(4000,7000)
    plt.grid(True, which="both", ls=":", color = 'gray', linewidth = 0.8)
    txt_prop = {'style' : 'oblique', 'family' : 'serif', 'size' :30}       
    plt.legend(title = f'{name[:-4]} \n Z = {Z} \n PLATEID = {plateid} \n MJD = {mjd} \n FIBERID = {fiberid}', prop = txt_prop, loc= 'upper right', title_fontsize=35)
    plt.savefig("/home/hollman/DataHII/HIIGalaxy_Spectra(Chavez2012)/Spectra_plots/" + (name.replace('.tex',"")) + ".png" ,
                bbox_inches='tight', transparent=False,dpi=150)
    #plt.show()
    plt.close('all')
    print('Done for '+ (name.replace('.tex',"")))
    gc.collect()
    del name, plateid, mjd, fiberid,z_finder,reduced_cat, Z, data
    
#spSpec-52339-0578-060.tex
#spec_plotter(spec_list[0])

In [ ]:
for i in range(len(spec_list)):
    spec_plotter(spec_list[i])

Done for spSpec-52339-0578-060
Done for spSpec-52937-1269-177
Done for spSpec-53297-1781-055
Done for spSpec-52962-1585-261
Done for spSpec-53473-1695-627
Done for spSpec-52353-0507-521
Done for spSpec-53534-2112-557
Done for spSpec-51929-0490-128
Done for spSpec-54561-2708-193
Done for spSpec-51821-0384-281
Done for spSpec-53768-2372-508
Done for spSpec-54628-2318-286
Done for spSpec-51882-0442-156
Done for spSpec-53386-1872-526
Done for spSpec-52174-0637-523
Done for spSpec-52224-0564-216
Done for spSpec-52636-0999-517
Done for spSpec-51957-0502-007
Done for spSpec-53084-1758-338
Done for spSpec-52413-0976-600
Done for spSpec-52178-0640-267
Done for spSpec-53463-1981-438
Done for spSpec-52641-1003-327
Done for spSpec-53768-2373-560
Done for spSpec-52939-1582-335
Done for spSpec-51816-0382-328
Done for spSpec-53317-1921-281
Done for spSpec-53142-1697-415
Done for spSpec-52138-0652-090
Done for spSpec-52238-0566-497
Done for spSpec-53786-2356-172
Done for spSpec-52174-0664-355
Done for